In [1]:
# %pip install requests
# %pip install beautifulsoup4
# %pip install pandas openpyxl

In [2]:
# !pip install bs4
# !pip install pandas

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [4]:
url = "https://www.magicbricks.com/property-for-sale/residential-real-estate?bedroom=2,3&cityName=Navi-Mumbai"

headers = {"User-Agent": "Mozilla/5.0"}   # to mimic a browser
responce = requests.get(url, headers=headers)

In [5]:
responce.status_code   # 200 = success

200

In [6]:
soup = BeautifulSoup(responce.text, "html.parser")

In [7]:
CARD_CLASS = "mb-srp__card"  

all_products = soup.find_all("div", class_=CARD_CLASS)
print("cards found:", len(all_products))

cards found: 30


In [8]:
if all_products:
    print(all_products[0].text.strip()[:600])

3 BHK Apartment for Sale in Maithili The Trellis, Kopar Khairane Navi MumbaiMaithili The Trellis Carpet Area1077 sqftUnder ConstructionPoss. by Mar '29TransactionNew PropertyFurnishingUnfurnishedSocietyMaithili The TrellisBathroom4Balcony4Multistorey Apartment for Sale in Kopar Khairane, Navi Mumbai. Covered area is 1077.0  Sq-ft. This property belongs to "The Trellis" .   View Property  NearbyKoparkhairane Railway StationKopar Khairane Railway StationMillennium Business ParkMillenium Business ParkNorth Point SchoolLokmanya Tilak College Of EngineeringChild Maternity Hospital Nirmal HospitalBa


In [9]:
title, price, area, bhk, bathroom, furnishing, floor, transaction, society, posted_by = [],[],[],[],[],[],[],[],[],[]

def grab(card, key):
    el = card.find(attrs={"data-summary": key})
    return el.text.strip() if el else None

for prod in all_products:
    try:
        title.append(prod.find("h2", class_="mb-srp__card--title").text.strip())
    except:
        title.append(None)
    try:
        price.append(prod.find("div", class_="mb-srp__card__price--amount").text.strip())
    except:
        price.append(None)
    area.append(grab(prod, "super-area") or grab(prod, "carpet-area"))
    bhk.append(grab(prod, "bedroom"))
    bathroom.append(grab(prod, "bathroom"))
    furnishing.append(grab(prod, "furnishing"))
    floor.append(grab(prod, "floor"))
    transaction.append(grab(prod, "transaction"))
    try:
        society.append(prod.find("div", class_="mb-srp__card__society").text.strip())
    except:
        society.append(None)
    try:
        posted_by.append(prod.find("div", class_="mb-srp__card__ads--name").text.strip())
    except:
        posted_by.append(None)

In [10]:
len(price)

30

In [11]:
data = {
    "title": title, "price": price, "area": area, "bhk": bhk,
    "bathroom": bathroom, "furnishing": furnishing, "floor": floor,
    "transaction": transaction, "society": society, "posted_by": posted_by,
}
df = pd.DataFrame(data)
df.shape

(30, 10)

In [12]:
import time

title, price, area, bhk, bathroom, furnishing, floor, transaction, society, posted_by = [],[],[],[],[],[],[],[],[],[]

base = "https://www.magicbricks.com/property-for-sale/residential-real-estate?bedroom=2,3&cityName=Navi-Mumbai"
PAGES = 25   

for i in range(1, PAGES + 1):
    url = f"{base}&page={i}"
    headers = {"User-Agent": "Mozilla/5.0"}
    responce = requests.get(url, headers=headers)
    soup = BeautifulSoup(responce.text, "html.parser")
    all_products = soup.find_all("div", class_=CARD_CLASS)

    for prod in all_products:
        try:
            title.append(prod.find("h2", class_="mb-srp__card--title").text.strip())
        except:
            title.append(None)
        try:
            price.append(prod.find("div", class_="mb-srp__card__price--amount").text.strip())
        except:
            price.append(None)
        area.append(grab(prod, "super-area") or grab(prod, "carpet-area"))
        bhk.append(grab(prod, "bedroom"))
        bathroom.append(grab(prod, "bathroom"))
        furnishing.append(grab(prod, "furnishing"))
        floor.append(grab(prod, "floor"))
        transaction.append(grab(prod, "transaction"))
        try:
            society.append(prod.find("div", class_="mb-srp__card__society").text.strip())
        except:
            society.append(None)
        try:
            posted_by.append(prod.find("div", class_="mb-srp__card__ads--name").text.strip())
        except:
            posted_by.append(None)

    print(f"{i} page fetched out of {PAGES}", end="\r")
    time.sleep(3)   # be polite between pages

25 page fetched out of 25

In [13]:
for li in [title, price, area, bhk, bathroom, furnishing, floor, transaction, society, posted_by]:
    print(len(li))

750
750
750
750
750
750
750
750
750
750


In [14]:
data_final = {
    "title": title, "price": price, "area": area, "bhk": bhk,
    "bathroom": bathroom, "furnishing": furnishing, "floor": floor,
    "transaction": transaction, "society": society, "posted_by": posted_by,
}
df_final = pd.DataFrame(data_final)
df_final

,title,price,area,bhk,bathroom,furnishing,floor,transaction,society,posted_by
0,3 BHK Apartment for Sale in Maithili The Trell...,₹2.71 Cr,Carpet Area1077 sqft,None,Bathroom4,FurnishingUnfurnished,NaN,TransactionNew Property,NaN,NaN
1,3 BHK Apartment for Sale in Ravriya Neelkanth ...,₹1.77 Cr,Carpet Area1745 sqft,None,Bathroom3,FurnishingUnfurnished,NaN,TransactionNew Property,NaN,NaN
2,"3 BHK Apartment for Sale in Codename Citadel, ...",₹2.95 Cr,Carpet Area1110 sqft,None,Bathroom2,FurnishingUnfurnished,NaN,TransactionNew Property,NaN,NaN
3,3 BHK Apartment for Sale in Pristine Bhaveshwa...,₹1.63 Cr,Super Area1795 sqft,None,Bathroom3,FurnishingUnfurnished,NaN,TransactionNew Property,NaN,NaN
4,2 BHK Apartment for Sale in Gajra Bhoomi Seren...,₹1.06 Cr,Carpet Area523 sqft,None,Bathroom2,FurnishingUnfurnished,NaN,TransactionNew Property,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
745,"3 BHK Apartment for Sale in Ambe Prerna, Ghans...",₹1.20 Cr,Carpet Area720 sqft,None,Bathroom2,FurnishingSemi-Furnished,Floor5 out of 7,TransactionResale,NaN,Owner: Rajan Singh
746,"2 BHK Apartment for Sale in Mamta Residency, T...",₹60 Lac,Carpet Area948 sqft,None,Bathroom2,FurnishingUnfurnished,Floor1 out of 7,TransactionResale,NaN,Owner: Rajpal Singh Bajwa
747,"2 BHK Apartment for Sale in Mamta Residency, T...",₹57 Lac,Carpet Area778 sqft,None,Bathroom2,FurnishingUnfurnished,Floor1 out of 7,TransactionResale,NaN,Owner: Rajnish Patel
748,3 BHK Apartment for Sale in Babas Pearl Height...,₹1.30 Cr,Carpet Area1152 sqft,None,Bathroom3,FurnishingUnfurnished,Floor1 out of 13,TransactionResale,NaN,Owner: RAJKUMAR GOEL


In [15]:
# df_final.to_csv("MagicBricks_NaviMumbai_raw.csv", index=False)
df_final.to_csv("MagicBricks_NaviMumbai_raw.csv", index=False, encoding="utf-8-sig")